# 4. Process PyNNLF Output: AEDP Aggregation Levels

Creates summary tables after the aggregation-level PyNNLF experiments have been run. This version expects 3 samples per aggregation level, `ds25` through `ds36`.


## 1. Setup And Paths

In [ ]:
from pathlib import Path
import sys

def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")
PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")
import pandas as pd
RESULTS_DIR = PROJECT_DIR / "results" / "03_aedp_aggregation_level"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RECAP_PATH = PROJECT_DIR / "experiment_result" / "a1_experiment_result.csv"
DATA_EXPLORATION_DIR = RESULTS_DIR / "01_data_exploration"
SAMPLE_DESIGN_PATH = DATA_EXPLORATION_DIR / "aedp_aggregation_sample_design.csv"
MODEL_ORDER = ["m1_naive_hp1", "m6_lr_hp1", "m17_xgb_hp1"]
FH8_MINUTES = 1440
SAMPLES_PER_LEVEL = 3
AGGREGATION_LEVELS = [1, 10, 100, 1000]
DATASET_IDS = [f"ds{i}" for i in range(25, 25 + SAMPLES_PER_LEVEL * len(AGGREGATION_LEVELS))]

## 2. Load Recap And Sample Design

In [ ]:
if not RECAP_PATH.exists():
    raise FileNotFoundError(f"Missing recap at {RECAP_PATH}. Run notebook 3 first.")
if not SAMPLE_DESIGN_PATH.exists():
    raise FileNotFoundError(f"Missing sample design at {SAMPLE_DESIGN_PATH}. Run notebook 2.2 first.")

recap = pd.read_csv(RECAP_PATH)
sample_design = pd.read_csv(SAMPLE_DESIGN_PATH)
expected_dataset_ids = DATASET_IDS
actual_design_ids = sample_design["dataset_id"].astype(str).tolist()
if actual_design_ids != expected_dataset_ids:
    raise ValueError(
        "Sample design does not match the expected 3-sample dataset range. "
        f"Expected {expected_dataset_ids[0]} through {expected_dataset_ids[-1]}, "
        f"found {actual_design_ids[:3]} ... {actual_design_ids[-3:]}"
    )
level_counts = sample_design.groupby("aggregation_level_hh")["dataset_id"].nunique().to_dict()
if level_counts != {level: SAMPLES_PER_LEVEL for level in AGGREGATION_LEVELS}:
    raise ValueError(f"Expected {SAMPLES_PER_LEVEL} samples per aggregation level, found {level_counts}")

rows = recap.loc[
    recap["dataset_no"].astype(str).isin(expected_dataset_ids)
    & pd.to_numeric(recap["forecast_horizon_min"], errors="coerce").eq(FH8_MINUTES)
].copy()
rows = rows.merge(sample_design, left_on="dataset_no", right_on="dataset_id", how="left")
if rows["aggregation_level_hh"].isna().any():
    missing_design = sorted(rows.loc[rows["aggregation_level_hh"].isna(), "dataset_no"].astype(str).unique())
    raise ValueError(f"Some recap rows did not match the sample design: {missing_design}")
rows["model_name"] = pd.Categorical(rows["model_name"], categories=MODEL_ORDER, ordered=True)
expected = {(dataset_id, model) for dataset_id in expected_dataset_ids for model in MODEL_ORDER}
actual = set(zip(rows["dataset_no"].astype(str), rows["model_name"].astype(str)))
missing = sorted(expected - actual)
if missing:
    raise ValueError(f"AEDP aggregation results are incomplete. Missing {len(missing)} combinations: {missing[:10]}")
if rows.shape[0] != len(expected):
    raise ValueError(f"Expected {len(expected)} recap rows, found {rows.shape[0]}")

display(rows[["dataset_no", "aggregation_level_hh", "sample_no", "model_name", "test_nRMSE", "test_nRMSE_stddev"]].head())


## 3. Build Summary Tables

In [ ]:
rows["runtime_s"] = pd.to_numeric(rows["runtime_ms"], errors="coerce") / 1000.0
summary = rows.groupby(["aggregation_level_hh", "model_name"], observed=False).agg(
    mean_test_nRMSE=("test_nRMSE", "mean"),
    sample_std_test_nRMSE=("test_nRMSE", "std"),
    mean_test_nRMSE_stddev=("test_nRMSE_stddev", "mean"),
    mean_runtime_s=("runtime_s", "mean"),
    n_samples=("dataset_no", "nunique"),
).reset_index()
wide_mean = summary.pivot(index="model_name", columns="aggregation_level_hh", values="mean_test_nRMSE").reindex(MODEL_ORDER)
wide_sample_std = summary.pivot(index="model_name", columns="aggregation_level_hh", values="sample_std_test_nRMSE").reindex(MODEL_ORDER)
wide_cv_stddev = summary.pivot(index="model_name", columns="aggregation_level_hh", values="mean_test_nRMSE_stddev").reindex(MODEL_ORDER)
wide_runtime = summary.pivot(index="model_name", columns="aggregation_level_hh", values="mean_runtime_s").reindex(MODEL_ORDER)
wide_completed = summary.pivot(index="model_name", columns="aggregation_level_hh", values="n_samples").reindex(MODEL_ORDER)

PAPER_MODEL_LABELS = {"m1_naive_hp1": "naive_hp1", "m6_lr_hp1": "lr_hp1", "m17_xgb_hp1": "xgb_hp1"}
paper_rows = []
for model in MODEL_ORDER:
    row = {"Model Name": PAPER_MODEL_LABELS.get(model, model)}
    for level in AGGREGATION_LEVELS:
        row[f"{level}hh Mean Test nRMSE (%)"] = wide_mean.loc[model, level]
        row[f"{level}hh Sample Std Test nRMSE (%)"] = wide_sample_std.loc[model, level]
        row[f"{level}hh Mean Test nRMSE Stddev (%)"] = wide_cv_stddev.loc[model, level]
        row[f"{level}hh Mean Training Time (s)"] = wide_runtime.loc[model, level]
        row[f"{level}hh Completed Samples"] = wide_completed.loc[model, level]
    paper_rows.append(row)
paper_table = pd.DataFrame(paper_rows)
for column in paper_table.columns.drop("Model Name"):
    if column.endswith("Completed Samples"):
        paper_table[column] = pd.to_numeric(paper_table[column], errors="coerce").astype("Int64")
    elif column.endswith("Training Time (s)"):
        paper_table[column] = pd.to_numeric(paper_table[column], errors="coerce").round(1)
    else:
        paper_table[column] = pd.to_numeric(paper_table[column], errors="coerce").round(2)

rows.to_csv(RESULTS_DIR / "aedp_aggregation_fh8_recap.csv", index=False)
summary.to_csv(RESULTS_DIR / "aedp_aggregation_fh8_summary_by_level_model.csv", index=False)
wide_mean.to_csv(RESULTS_DIR / "aedp_aggregation_fh8_mean_nrmse_by_level.csv")
wide_sample_std.to_csv(RESULTS_DIR / "aedp_aggregation_fh8_sample_std_nrmse_by_level.csv")
wide_cv_stddev.to_csv(RESULTS_DIR / "aedp_aggregation_fh8_mean_cv_stddev_by_level.csv")
wide_runtime.to_csv(RESULTS_DIR / "aedp_aggregation_fh8_mean_runtime_seconds_by_level.csv")
paper_table.to_csv(RESULTS_DIR / "paper_table_aedp_aggregation_levels.csv", index=False)

display(summary.round(3))
display(wide_mean.round(3))
display(paper_table)